In [ ]:
# Standard Python system libraries
import os
import re
import json
import warnings

# Library for saving and loading trained models
import joblib

# Core math and data manipulation libraries
import numpy as np
import pandas as pd

# Visualization and charting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn: Data splitting and preprocessing tools
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Scikit-Learn: Machine learning classification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Scikit-Learn: Evaluation metrics to grade our models
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve

# Imbalanced-Learn: Tools for balancing datasets
from imblearn.pipeline import Pipeline as ImbalancedPipeline
from imblearn.over_sampling import SMOTE

# Turn off unnecessary warning messages
warnings.filterwarnings('ignore')

# Set a clean white-grid style for all our charts
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

print("Step 1 Complete: All necessary libraries imported successfully!")


In [ ]:
# List of possible places where the dataset CSV might be located
candidate_file_locations = [
    os.path.join("dataset", "liver_disease_data.csv"),
    os.path.join("dataset", "Liver Patient Dataset (LPD)_train.csv"),
    "Liver Patient Dataset (LPD)_train.csv",
    "../dataset/liver_disease_data.csv",
    "../dataset/Liver Patient Dataset (LPD)_train.csv"
]

# Look for the dataset file one location at a time
dataset_file_path = None
for file_path in candidate_file_locations:
    if os.path.exists(file_path):
        dataset_file_path = file_path
        break

# If the file was not found anywhere, raise an informative error
if dataset_file_path is None:
    raise FileNotFoundError("Could not find the dataset CSV file. Please make sure it is in the project folder.")

print(f"Dataset successfully located at: {dataset_file_path}")

# Load the dataset using latin1 encoding to avoid character errors
try:
    raw_patient_data = pd.read_csv(dataset_file_path, encoding='latin1')
except Exception:
    raw_patient_data = pd.read_csv(dataset_file_path, encoding='utf-8')

# Display the size of the dataset
total_rows = raw_patient_data.shape[0]
total_columns = raw_patient_data.shape[1]
print(f"Total Patient Records: {total_rows:,} rows")
print(f"Total Columns: {total_columns} columns")

print("\nList of Raw Columns in the CSV:")
for column_index, column_name in enumerate(raw_patient_data.columns):
    column_datatype = raw_patient_data[column_name].dtype
    print(f"  Column {column_index}: {repr(column_name)} (Type: {column_datatype})")

print("\nPreview of the first 5 rows:")
raw_patient_data.head(5)


In [ ]:
# A clear dictionary mapping messy raw names to clean standard names
COLUMN_NAME_DICTIONARY = {
    "age of the patient": "age",
    "gender of the patient": "gender",
    "total bilirubin": "total_bilirubin",
    "direct bilirubin": "direct_bilirubin",
    "alkphos alkaline phosphotase": "alkaline_phosphotase",
    "sgpt alamine aminotransferase": "alamine_aminotransferase",
    "sgot aspartate aminotransferase": "aspartate_aminotransferase",
    "total protiens": "total_proteins",
    "total proteins": "total_proteins",
    "alb albumin": "albumin",
    "a/g ratio albumin and globulin ratio": "albumin_globulin_ratio",
    "result": "result"
}

def clean_single_column_name(raw_name):
    # Step 1: Replace non-breaking space character with a normal space
    name_clean = raw_name.replace('\xa0', ' ')
    # Step 2: Remove leading and trailing spaces
    name_clean = name_clean.strip()
    # Step 3: Convert all characters to lowercase
    name_clean = name_clean.lower()
    # Step 4: Collapse any multiple spaces into one single space
    name_clean = re.sub(r'\s+', ' ', name_clean)
    
    # Step 5: If the name is in our dictionary, return the standard clean name
    if name_clean in COLUMN_NAME_DICTIONARY:
        return COLUMN_NAME_DICTIONARY[name_clean]
    else:
        return name_clean

# Make a working copy of the dataset
patient_data = raw_patient_data.copy()

# Use a simple, readable for loop to clean each column name
cleaned_column_list = []
for column_name in patient_data.columns:
    new_clean_name = clean_single_column_name(column_name)
    cleaned_column_list.append(new_clean_name)

# Assign the clean names back to our DataFrame
patient_data.columns = cleaned_column_list

# Also clean the text inside the 'gender' column (e.g. capitalize 'Male' and 'Female')
if 'gender' in patient_data.columns:
    patient_data['gender'] = patient_data['gender'].astype(str).str.strip().str.title()
    # Replace text 'Nan' or 'None' with true missing values (np.nan)
    patient_data['gender'] = patient_data['gender'].replace({'Nan': np.nan, 'None': np.nan, '': np.nan})

print("Standardized Column Names:")
for col in patient_data.columns:
    print(f"  -> {col}")

patient_data.head(5)


In [ ]:
# Inspect the raw counts of the target before mapping
print("Target counts before mapping:")
print(patient_data['result'].value_counts())

# Filter out any invalid numbers if they exist (only keep 1 and 2)
patient_data = patient_data[patient_data['result'].isin([1, 2])]
patient_data = patient_data.reset_index(drop=True)

# Map the numbers: 1 stays 1 (Liver Disease), 2 becomes 0 (Healthy)
diagnosis_mapping = {1: 1, 2: 0}
patient_data['result'] = patient_data['result'].map(diagnosis_mapping)
patient_data['result'] = patient_data['result'].astype(int)

print("\nTarget counts after mapping:")
print(patient_data['result'].value_counts())
print("\nMapping explanation:")
print("  Label 1 = Patient with Liver Disease")
print("  Label 0 = Healthy Patient (No Liver Disease)")


In [ ]:
# Count how many duplicate rows exist
number_of_duplicate_rows = patient_data.duplicated().sum()
total_patient_count = len(patient_data)
duplicate_percentage = (number_of_duplicate_rows / total_patient_count) * 100

print(f"Total Rows Before Deduplication: {total_patient_count:,}")
print(f"Duplicate Rows Found: {number_of_duplicate_rows:,} ({duplicate_percentage:.2f}%)")

# Remove duplicate rows to prevent train-test data leakage
cleaned_patient_data = patient_data.drop_duplicates()
cleaned_patient_data = cleaned_patient_data.reset_index(drop=True)

unique_patient_count = len(cleaned_patient_data)
print(f"Total Unique Patient Records Remaining: {unique_patient_count:,}")

# Look at the disease breakdown in the deduplicated data
sick_patient_count = (cleaned_patient_data['result'] == 1).sum()
healthy_patient_count = (cleaned_patient_data['result'] == 0).sum()

sick_percentage = (sick_patient_count / unique_patient_count) * 100
healthy_percentage = (healthy_patient_count / unique_patient_count) * 100

print("\nClass Distribution in Cleaned Data:")
print(f"  Liver Disease Patients (1): {sick_patient_count:,} ({sick_percentage:.2f}%)")
print(f"  Healthy Patients (0):       {healthy_patient_count:,} ({healthy_percentage:.2f}%)")


In [ ]:
print("Checking for missing values in each column:\n")

missing_summary_list = []
for column_name in cleaned_patient_data.columns:
    missing_count = cleaned_patient_data[column_name].isnull().sum()
    if missing_count > 0:
        missing_pct = (missing_count / len(cleaned_patient_data)) * 100
        print(f"  Column '{column_name}': {missing_count} missing values ({missing_pct:.2f}%)")
        missing_summary_list.append((column_name, missing_count, missing_pct))

if len(missing_summary_list) == 0:
    print("  No missing values found in any column!")
else:
    print(f"\nTotal columns with missing values: {len(missing_summary_list)}")


In [ ]:
# Create a figure with 2 subplots side-by-side
figure, (axis_bar, axis_donut) = plt.subplots(nrows=1, ncols=2, figsize=(13, 5))

category_labels = ["Liver Disease (1)", "Healthy (0)"]
category_counts = [sick_patient_count, healthy_patient_count]
bar_colors = ["#e74c3c", "#2ecc71"]  # Red for disease, Green for healthy

# Subplot 1: Bar Chart
bar_rectangles = axis_bar.bar(
    category_labels,
    category_counts,
    color=bar_colors,
    width=0.45,
    edgecolor="black"
)
axis_bar.set_title("Patient Count by Diagnosis", fontsize=13, fontweight="bold")
axis_bar.set_ylabel("Number of Patients", fontsize=11)
axis_bar.grid(axis="y", linestyle="--", alpha=0.7)

# Add clear count labels on top of each bar
for bar in bar_rectangles:
    bar_height = bar.get_height()
    pct_value = (bar_height / unique_patient_count) * 100
    axis_bar.annotate(
        f"{bar_height:,}\n({pct_value:.1f}%)",
        xy=(bar.get_x() + bar.get_width() / 2, bar_height / 2),
        ha="center",
        va="center",
        color="white",
        fontweight="bold",
        fontsize=11
    )

# Subplot 2: Donut Chart
axis_donut.pie(
    category_counts,
    labels=category_labels,
    colors=bar_colors,
    autopct="%1.1f%%",
    startangle=140,
    wedgeprops={"width": 0.4, "edgecolor": "white", "linewidth": 2}
)
axis_donut.set_title("Class Proportion Breakdown", fontsize=13, fontweight="bold")

plt.suptitle("Target Class Distribution: Liver Disease vs Healthy", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# List of all numerical biomarker columns
numerical_biomarker_columns = [
    "age",
    "total_bilirubin",
    "direct_bilirubin",
    "alkaline_phosphotase",
    "alamine_aminotransferase",
    "aspartate_aminotransferase",
    "total_proteins",
    "albumin",
    "albumin_globulin_ratio"
]

# Calculate correlation table including the target result
correlation_matrix = cleaned_patient_data[numerical_biomarker_columns + ['result']].corr()

# Create a triangular mask so we don't repeat the top half of the mirror
triangle_mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

# Draw the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    mask=triangle_mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1.0,
    vmax=1.0,
    square=True,
    linewidths=0.5
)
plt.title("Correlation Matrix of Hepatic Biomarkers & Diagnosis", fontsize=13, fontweight="bold", pad=15)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Set up a 3x3 grid of subplots (9 charts total)
figure, grid_axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 12))
flat_axes_list = grid_axes.flatten()

for index, biomarker_name in enumerate(numerical_biomarker_columns):
    current_axis = flat_axes_list[index]
    
    # Draw distribution curve for healthy (0) and sick (1)
    sns.kdeplot(
        data=cleaned_patient_data,
        x=biomarker_name,
        hue='result',
        common_norm=False,
        palette={1: "#e74c3c", 0: "#2ecc71"},
        ax=current_axis,
        fill=True,
        alpha=0.3
    )
    
    current_axis.set_title(f"Distribution: {biomarker_name}", fontsize=11, fontweight="bold")
    current_axis.set_ylabel("Density", fontsize=9)
    
    # If the numbers get very large (e.g. enzyme values > 100), use log scale for clarity
    if cleaned_patient_data[biomarker_name].max() > 100:
        current_axis.set_xscale("log")
        current_axis.set_xlabel(f"{biomarker_name} (Log Scale)", fontsize=9)
    else:
        current_axis.set_xlabel(biomarker_name, fontsize=9)

plt.suptitle("Biomarker Distributions by Diagnosis (Green = Healthy, Red = Diseased)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Define the official list of 10 features our models will use
ALL_FEATURE_COLUMNS = [
    "age",
    "gender",
    "total_bilirubin",
    "direct_bilirubin",
    "alkaline_phosphotase",
    "alamine_aminotransferase",
    "aspartate_aminotransferase",
    "total_proteins",
    "albumin",
    "albumin_globulin_ratio"
]

NUMERICAL_FEATURE_LIST = [
    "age",
    "total_bilirubin",
    "direct_bilirubin",
    "alkaline_phosphotase",
    "alamine_aminotransferase",
    "aspartate_aminotransferase",
    "total_proteins",
    "albumin",
    "albumin_globulin_ratio"
]

CATEGORICAL_FEATURE_LIST = ["gender"]

# Separate features X and target y
X = cleaned_patient_data[ALL_FEATURE_COLUMNS].copy()
y = cleaned_patient_data['result'].copy()

print(f"Features X shape: {X.shape} (10 patient biomarker columns)")
print(f"Target y shape:   {y.shape} (1 diagnosis answer column)")


In [ ]:
# Split data into 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training Set (80%):  {X_train.shape[0]:,} patient records")
print(f"Test Set (20%):      {X_test.shape[0]:,} patient records (unseen held-out exam)")

train_disease_pct = (y_train == 1).mean() * 100
test_disease_pct = (y_test == 1).mean() * 100

print(f"\nDisease proportion in Training Set: {train_disease_pct:.2f}%")
print(f"Disease proportion in Test Set:     {test_disease_pct:.2f}%")
print("Notice how 'stratify=y' kept the exact same disease proportion in both sets!")


In [ ]:
def create_data_preprocessor():
    # Pipeline for the 9 numerical biomarker columns
    numerical_steps = Pipeline(steps=[
        ("median_imputer", SimpleImputer(strategy="median")),
        ("standard_scaler", StandardScaler())
    ])
    
    # Pipeline for the 1 categorical column ('gender')
    categorical_steps = Pipeline(steps=[
        ("mode_imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot_encoder", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
    ])
    
    # Combine both into one master ColumnTransformer
    master_preprocessor = ColumnTransformer(
        transformers=[
            ("numerical_pipeline", numerical_steps, NUMERICAL_FEATURE_LIST),
            ("categorical_pipeline", categorical_steps, CATEGORICAL_FEATURE_LIST)
        ],
        remainder="drop"
    )
    return master_preprocessor

# Test our preprocessor on the training data
sample_preprocessor = create_data_preprocessor()
transformed_training_matrix = sample_preprocessor.fit_transform(X_train)

print(f"Original Training Features Shape:    {X_train.shape}")
print(f"Transformed Training Features Shape: {transformed_training_matrix.shape}")
print("Step 12 Complete: ColumnTransformer successfully constructed and verified!")


In [ ]:
# Define the 3 candidate machine learning models with fixed random_state for reproducibility
models_to_train = {
    "Logistic Regression": LogisticRegression(
        max_iter=1500,
        C=1.0,
        solver="lbfgs",
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
}

# Dictionaries to store our trained pipelines and their evaluation scores
trained_model_pipelines = {}
model_evaluation_metrics = {}

print("=" * 65)
print("TRAINING MODELS WITH SMOTE (APPLIED ONLY ON TRAINING DATA)")
print("=" * 65)

for model_name, classifier_algorithm in models_to_train.items():
    print(f"\n---> Currently Training: {model_name}...")
    
    # Build complete end-to-end pipeline: Preprocessor -> SMOTE -> Classifier
    complete_pipeline = ImbalancedPipeline(steps=[
        ("data_preprocessor", create_data_preprocessor()),
        ("smote_balancer", SMOTE(random_state=42)),
        ("classifier", classifier_algorithm)
    ])
    
    # Step A: Train the pipeline on training data
    complete_pipeline.fit(X_train, y_train)
    
    # Step B: Make predictions on the unseen test set
    predicted_labels = complete_pipeline.predict(X_test)
    
    # Step C: Get predicted probabilities if supported
    if hasattr(classifier_algorithm, "predict_proba"):
        predicted_probabilities = complete_pipeline.predict_proba(X_test)[:, 1]
        area_under_curve = roc_auc_score(y_test, predicted_probabilities)
    else:
        predicted_probabilities = None
        area_under_curve = 0.0

    # Step D: Calculate standard evaluation metrics
    accuracy = accuracy_score(y_test, predicted_labels)
    precision = precision_score(y_test, predicted_labels, zero_division=0)
    recall = recall_score(y_test, predicted_labels, zero_division=0)
    f1 = f1_score(y_test, predicted_labels, zero_division=0)
    
    conf_matrix = confusion_matrix(y_test, predicted_labels)
    true_negative, false_positive, false_negative, true_positive = conf_matrix.ravel()
    specificity = true_negative / (true_negative + false_positive) if (true_negative + false_positive) > 0 else 0.0
    
    # Save metrics in our dictionary
    model_evaluation_metrics[model_name] = {
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall (Sensitivity)": round(recall, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(area_under_curve, 4),
        "Specificity": round(specificity, 4),
        "True_Positive": int(true_positive),
        "False_Positive": int(false_positive),
        "True_Negative": int(true_negative),
        "False_Negative": int(false_negative),
        "Confusion_Matrix": conf_matrix.tolist()
    }
    
    trained_model_pipelines[model_name] = complete_pipeline
    
    print(f"  Accuracy:    {accuracy * 100:.2f}%")
    print(f"  Precision:   {precision * 100:.2f}%")
    print(f"  Recall:      {recall * 100:.2f}% (Sensitivity - catching sick patients)")
    print(f"  F1-Score:    {f1:.4f}")
    print(f"  ROC-AUC:     {area_under_curve:.4f}")
    print(f"  Specificity: {specificity * 100:.2f}%")


In [ ]:
# Create a clean DataFrame from our metrics dictionary
metrics_columns = ["Accuracy", "Precision", "Recall (Sensitivity)", "F1-Score", "ROC-AUC", "Specificity"]
benchmark_table = pd.DataFrame(model_evaluation_metrics).T[metrics_columns]

print("=" * 60)
print("TEST SET BENCHMARK TABLE (EVALUATED ON 3,874 UNSEEN PATIENTS)")
print("=" * 60)
benchmark_table


In [ ]:
# Chart 1: Bar Chart comparing metrics across models
chart_figure, chart_axis = plt.subplots(figsize=(11, 5))
benchmark_table[["Accuracy", "Precision", "Recall (Sensitivity)", "F1-Score", "ROC-AUC"]].plot(
    kind="bar",
    ax=chart_axis,
    width=0.8,
    colormap="viridis"
)
chart_axis.set_title("Machine Learning Model Performance on Unseen Test Set", fontsize=13, fontweight="bold", pad=15)
chart_axis.set_ylabel("Score (0.0 to 1.0)", fontsize=11)
chart_axis.set_xlabel("Model Architecture", fontsize=11)
chart_axis.set_ylim(0, 1.1)
chart_axis.tick_params(axis="x", rotation=0, labelsize=10)
chart_axis.legend(loc="lower right", frameon=True)

# Add exact score values on top of bars
for patch in chart_axis.patches:
    h = patch.get_height()
    if h > 0.05:
        chart_axis.annotate(
            f"{h:.2f}",
            (patch.get_x() + patch.get_width() / 2.0, h),
            ha='center', va='bottom', fontsize=8, xytext=(0, 2), textcoords='offset points'
        )

plt.tight_layout()
plt.show()


In [ ]:
# Chart 2: Side-by-Side Confusion Matrices
fig_cm, axes_cm = plt.subplots(nrows=1, ncols=3, figsize=(16, 5))

for ax, (name, metrics) in zip(axes_cm, model_evaluation_metrics.items()):
    matrix_data = np.array(metrics["Confusion_Matrix"])
    sns.heatmap(
        matrix_data,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=["Healthy (0)", "Liver Disease (1)"],
        yticklabels=["Healthy (0)", "Liver Disease (1)"]
    )
    rec_val = metrics['Recall (Sensitivity)'] * 100
    spec_val = metrics['Specificity'] * 100
    ax.set_title(f"{name}\nRecall: {rec_val:.1f}% | Specificity: {spec_val:.1f}%", fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted Diagnosis", fontsize=10)
    ax.set_ylabel("Actual Diagnosis", fontsize=10)

plt.suptitle("Confusion Matrices on Held-Out Test Set (3,874 Patients)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Chart 3: Receiver Operating Characteristic (ROC) Curves
plt.figure(figsize=(8, 6))

for name, pipeline in trained_model_pipelines.items():
    classifier_step = pipeline.named_steps["classifier"]
    if hasattr(classifier_step, "predict_proba"):
        probabilities = pipeline.predict_proba(X_test)[:, 1]
        false_positive_rate, true_positive_rate, _ = roc_curve(y_test, probabilities)
        area_score = roc_auc_score(y_test, probabilities)
        plt.plot(false_positive_rate, true_positive_rate, lw=2, label=f"{name} (AUC = {area_score:.4f})")

plt.plot([0, 1], [0, 1], color="grey", lw=1.5, linestyle="--", label="Random Guessing (AUC = 0.500)")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
plt.ylabel("True Positive Rate (Recall / Sensitivity)", fontsize=11)
plt.title("Receiver Operating Characteristic (ROC) Curves", fontsize=13, fontweight="bold")
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# Calculate composite clinical scores for each model
best_model_name = None
highest_composite_score = -1.0

print("=" * 65)
print("CLINICAL COMPOSITE SCORING (PRIORITIZING RECALL FOR MEDICAL SAFETY)")
print("Composite Score = 0.40 * Recall + 0.35 * F1 + 0.25 * ROC-AUC")
print("=" * 65)

for model_name, metrics in model_evaluation_metrics.items():
    recall_score_val = metrics["Recall (Sensitivity)"]
    f1_score_val = metrics["F1-Score"]
    auc_score_val = metrics["ROC-AUC"]
    
    composite_score = (0.40 * recall_score_val) + (0.35 * f1_score_val) + (0.25 * auc_score_val)
    print(f"  {model_name:20s}: Composite Score = {composite_score:.4f}")
    
    if composite_score > highest_composite_score:
        highest_composite_score = composite_score
        best_model_name = model_name

print(f"\n>>> Winning Model Selected: {best_model_name} (Composite Score: {highest_composite_score:.4f})")

# Save our trained pipeline to disk
os.makedirs("models", exist_ok=True)
saved_model_path = os.path.join("models", "beginner_friendly_model.joblib")
winning_pipeline = trained_model_pipelines[best_model_name]

joblib.dump(winning_pipeline, saved_model_path)
print(f"[OK] Trained pipeline successfully saved to: {saved_model_path}")


In [ ]:
# Physiological reference ranges for safety checks
NORMAL_PHYSIOLOGICAL_BOUNDS = {
    "age": {"min": 1, "max": 120, "label": "Age (years)"},
    "gender": {"allowed": ["Male", "Female"], "label": "Biological Sex"},
    "total_bilirubin": {"min": 0.1, "max": 80.0, "label": "Total Bilirubin (mg/dL)"},
    "direct_bilirubin": {"min": 0.05, "max": 30.0, "label": "Direct Bilirubin (mg/dL)"},
    "alkaline_phosphotase": {"min": 20.0, "max": 3000.0, "label": "Alkaline Phosphatase (ALP IU/L)"},
    "alamine_aminotransferase": {"min": 5.0, "max": 3000.0, "label": "ALT / SGPT (IU/L)"},
    "aspartate_aminotransferase": {"min": 5.0, "max": 5000.0, "label": "AST / SGOT (IU/L)"},
    "total_proteins": {"min": 1.0, "max": 15.0, "label": "Total Proteins (g/dL)"},
    "albumin": {"min": 0.5, "max": 8.0, "label": "Albumin (g/dL)"},
    "albumin_globulin_ratio": {"min": 0.1, "max": 5.0, "label": "A/G Ratio"}
}

def validate_patient_lab_inputs(patient_dictionary):
    validation_error_list = []
    
    # Check 1: Ensure all 10 required features are present
    for feature in ALL_FEATURE_COLUMNS:
        if feature not in patient_dictionary or patient_dictionary[feature] is None:
            validation_error_list.append(f"Missing required test result: '{feature}'")
            
    if len(validation_error_list) > 0:
        return False, validation_error_list
        
    # Check 2: Gender must be Male or Female
    clean_gender = str(patient_dictionary["gender"]).strip().title()
    if clean_gender not in ["Male", "Female"]:
        validation_error_list.append(f"Invalid gender '{patient_dictionary['gender']}'. Must be 'Male' or 'Female'.")
        
    # Check 3: Physiological ranges
    for feature_name, bounds in NORMAL_PHYSIOLOGICAL_BOUNDS.items():
        if feature_name == "gender":
            continue
        try:
            numeric_value = float(patient_dictionary[feature_name])
            if numeric_value < bounds["min"] or numeric_value > bounds["max"]:
                validation_error_list.append(
                    f"{bounds['label']} ({numeric_value}) is outside plausible range [{bounds['min']}, {bounds['max']}]."
                )
        except (ValueError, TypeError):
            validation_error_list.append(f"'{feature_name}' must be a valid number.")
            
    # Check 4: Domain Rule - Direct Bilirubin cannot exceed Total Bilirubin
    try:
        total_b = float(patient_dictionary.get("total_bilirubin", 0))
        direct_b = float(patient_dictionary.get("direct_bilirubin", 0))
        if direct_b > total_b + 0.1:
            validation_error_list.append(
                f"Direct Bilirubin ({direct_b} mg/dL) cannot exceed Total Bilirubin ({total_b} mg/dL)."
            )
    except Exception:
        pass
        
    is_valid = (len(validation_error_list) == 0)
    return is_valid, validation_error_list


def predict_new_patient(patient_dictionary, model_file_path=saved_model_path):
    # Step 1: Validate inputs
    is_valid, error_messages = validate_patient_lab_inputs(patient_dictionary)
    if not is_valid:
        return {
            "success": False,
            "errors": error_messages,
            "prediction": None,
            "label": None,
            "probability": None
        }
        
    # Step 2: Load the saved model pipeline
    loaded_model = joblib.load(model_file_path)
    
    # Step 3: Put input into a single-row DataFrame
    single_patient_df = pd.DataFrame([{col: patient_dictionary[col] for col in ALL_FEATURE_COLUMNS}])
    single_patient_df["gender"] = single_patient_df["gender"].astype(str).str.strip().str.title()
    for col in NUMERICAL_FEATURE_LIST:
        single_patient_df[col] = pd.to_numeric(single_patient_df[col], errors="coerce")
        
    # Step 4: Run prediction
    predicted_class = int(loaded_model.predict(single_patient_df)[0])
    
    # Step 5: Get probability
    if hasattr(loaded_model, "predict_proba"):
        disease_probability = float(loaded_model.predict_proba(single_patient_df)[0][1])
    else:
        disease_probability = 1.0 if predicted_class == 1 else 0.0
        
    # Step 6: Assign risk tier
    if disease_probability < 0.30:
        risk_category = "Low Risk"
    elif disease_probability < 0.65:
        risk_category = "Moderate / Borderline Risk"
    else:
        risk_category = "High Risk"
        
    diagnosis_label = "Liver Disease Detected" if predicted_class == 1 else "Healthy / No Liver Disease Detected"
    
    return {
        "success": True,
        "prediction": predicted_class,
        "label": diagnosis_label,
        "probability": round(disease_probability, 4),
        "risk_level": risk_category,
        "disclaimer": "[NOTICE]: This is an experimental machine learning estimate for educational purposes, not a clinical diagnosis."
    }

print("Step 17 Complete: Safe prediction function ready!")




### 1. What this code does:
We test our prediction function on three realistic scenarios:
1. **Case 1 (High-Risk Patient):** A 62-year-old male with high Bilirubin (10.9) and high enzymes (ALT 64, AST 100).
2. **Case 2 (Healthy Patient):** A 28-year-old female with completely normal lab tests.
3. **Case 3 (Invalid Input):** Missing tests and impossible values.

### 2. Expected Results:
* Case 1 should predict **Liver Disease Detected** with a high probability (>95%).
* Case 2 should predict **Healthy / No Liver Disease Detected** with low risk.
* Case 3 should cleanly report error messages and refuse to make a blind guess.


In [ ]:
# Case 1: Sick Patient Profile (Acute hepatic stress)
sick_patient_sample = {
    "age": 62,
    "gender": "Male",
    "total_bilirubin": 10.9,
    "direct_bilirubin": 5.5,
    "alkaline_phosphotase": 699,
    "alamine_aminotransferase": 64,
    "aspartate_aminotransferase": 100,
    "total_proteins": 7.5,
    "albumin": 3.2,
    "albumin_globulin_ratio": 0.74
}

result_case1 = predict_new_patient(sick_patient_sample)
print("=== CASE 1: SICK PATIENT SAMPLE ===")
print("  Predicted Diagnosis: ", result_case1["label"])
print(f"  Disease Probability:  {result_case1['probability'] * 100:.2f}%")
print("  Risk Category:        ", result_case1["risk_level"])
print("  Advisory:             ", result_case1["disclaimer"])

# Case 2: Healthy Patient Profile (Normal blood tests)
healthy_patient_sample = {
    "age": 28,
    "gender": "Female",
    "total_bilirubin": 0.7,
    "direct_bilirubin": 0.1,
    "alkaline_phosphotase": 130,
    "alamine_aminotransferase": 18,
    "aspartate_aminotransferase": 20,
    "total_proteins": 7.1,
    "albumin": 4.2,
    "albumin_globulin_ratio": 1.35
}

result_case2 = predict_new_patient(healthy_patient_sample)
print("\n=== CASE 2: HEALTHY PATIENT SAMPLE ===")
print("  Predicted Diagnosis: ", result_case2["label"])
print(f"  Disease Probability:  {result_case2['probability'] * 100:.2f}%")
print("  Risk Category:        ", result_case2["risk_level"])

# Case 3: Invalid Input Test (Impossible Age & Missing Tests)
invalid_patient_sample = {
    "age": 220,  # Impossible age
    "gender": "Male",
    "total_bilirubin": 1.0,
    "direct_bilirubin": 6.0  # Direct > Total (impossible)
}

result_case3 = predict_new_patient(invalid_patient_sample)
print("\n=== CASE 3: INVALID INPUT TEST ===")
print("  Success: ", result_case3["success"])
print("  Errors Caught:")
for err in result_case3["errors"]:
    print(f"   -> {err}")
